# Exercise 2: PyTorch core

In this exercise you’ll build core PyTorch “muscle memory” that you’ll reuse in basically every model you write:

- **Autograd**: how gradients are created, how they accumulate, and how to compute gradients for one or multiple inputs.
- **Dataloading**: writing small `Dataset`s, using `DataLoader`, and custom `collate_fn`.
- **Optimizers**: implementing **AdamW** updates from scratch (state, bias correction, weight decay).
- **Training basics**: a clean single training step.
- **Initialization**: fan-in/out and common initializers (Xavier / Kaiming), plus a helper to init `nn.Linear`.

As before: fill in all `TODO`s without changing function names or signatures.
When debugging, print shapes/dtypes/devices, and write tiny sanity checks (e.g. compare to PyTorch’s built-ins).


In [20]:
from __future__ import annotations
from dataclasses import dataclass
import torch
from torch import nn

## Autograd fundamentals

PyTorch builds a computation graph when you apply operations to tensors with `requires_grad=True`.
Calling `backward()` (or `torch.autograd.grad`) computes gradients by traversing that graph.

### Key concepts
- **Leaf tensor**: a tensor created by you (not the result of an operation) with `requires_grad=True`. Leaf tensors can store gradients in `.grad`.
- **Gradient accumulation**: calling `backward()` adds into `.grad` (it does not overwrite). You must reset gradients between steps/calls.
- **`torch.autograd.grad` vs `.backward()`**
  - `torch.autograd.grad(f, x)` returns `df/dx` directly and does not write into `x.grad` unless you explicitly do so.
  - `f.backward()` writes gradients into `.grad` of leaf tensors.

In the next functions you’ll compute gradients for a simple scalar function such as `f(x) = sum(x^2)` using both APIs.

### `torch.no_grad()`
Wrap inference-only code to avoid tracking gradients and building graphs:
- saves memory
- speeds up evaluation

### `detach()`
`y = x.detach()` returns a tensor that shares data with `x` but is **not connected** to the autograd graph.
This is useful when you want to treat something as a constant target.

### `model.train()` vs `model.eval()`
- `train()` enables training behavior (e.g. dropout active, batchnorm updates running stats).让 Dropout 随机丢弃、BatchNorm 用当前 batch 统计量并更新 running stats。
- `eval()` enables inference behavior (e.g. dropout off, batchnorm uses running stats).让 Dropout 关闭、BatchNorm 用累积的 running stats 且不再更新。


In [21]:
def grad_with_autograd_grad(x: torch.Tensor) -> torch.Tensor:
    """
    Compute gradient of f(x) = sum(x^2) using torch.autograd.grad

    Requirements:
    - Do not call .backward().
    - x should require grad inside the function (don't assume it does).
    - Must return df/dx
    """
    # TODO: implement
    x_local = x.detach().requires_grad_(True) # 方法（带下划线）,就地(in-place)把该张量的 requires_grad 设为 True。
    f = x_local.square().sum()
    (grad,)=torch.autograd.grad(f,x_local)
    return grad
    raise NotImplementedError

In [22]:

def grad_with_backward(x: torch.Tensor) -> torch.Tensor:
    """
    Compute gradient of f(x) = sum(x^2) using .backward().

    Requirements:
    - Must return df/dx
    - Must not leak gradients across calls (watch x.grad accumulation)
    """
    # TODO: implement
    x_local = x.detach().requires_grad_(True)
    f = x_local.square().sum()
    f.backward()
    return x_local.grad.clone() # 生成一份独立副本，避免后续对该张量或它的 .grad 做操作时，意外修改到已经返回的梯度。
    raise NotImplementedError

In [23]:
def grad_wrt_multiple_inputs(
    a: torch.Tensor, b: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Compute gradients w.r.t.(with respeact to) multiple inputs. The function is f(a, b) = sum(a^2 + ab).

    Return:
        (df/da, df/db)

    Requirements:
    - Use torch.autograd.grad
    - Ensure both a and b require grad in this function.
    """
    # TODO: implement
    a_local = a.detach().requires_grad_(True)
    b_local = b.detach().requires_grad_(True)
    f = (a_local.square()+a_local*b_local).sum()
    grad_a,grad_b = torch.autograd.grad(f,(a_local,b_local))
    return grad_a,grad_b
    raise NotImplementedError

## Dataloading

In PyTorch, a `Dataset` defines how to fetch a *single* training example, and a `DataLoader` handles:
- batching
- shuffling
- parallel workers
- optional custom batching logic via `collate_fn`

### `Dataset` in one sentence
A `Dataset` only needs:
- `__len__`: number of items
- `__getitem__`: return one item (e.g. `(x, y)`)

### Why `collate_fn` matters
The default DataLoader collation stacks items along a new batch dimension.
That works for fixed-size tensors, but it breaks for **variable-length sequences**.

So we’ll implement padding ourselves:
- Convert a list of 1D token sequences into a padded tensor `(B, T_max)`
- Track `lengths` and a `padding_mask`

### Mask convention for padding
For padding masks in this exercise:
- `padding_mask[b, t] == True` means **this is padding / invalid**
- `padding_mask[b, t] == False` means **this is a real token**

In [24]:
from torch.utils.data import DataLoader, Dataset

In [ ]:
class TensorPairDataset(Dataset):
    """
    Minimal dataset wrapping (x, y).

    x: (N, ...)
    y: (N, ...)

    N is the number of samples. The dataset should return tuples of (x[i], y[i]).
    """

    def __init__(self, x: torch.Tensor, y: torch.Tensor):
        # TODO: implement
        if x.ndim < 1 or y.ndim < 1 or x.shape[0] != y.shape[0]:
            raise ValueError("x and y must have the same sample dimension")
        self.x = x
        self.y = y

    def __len__(self) -> int:
        # TODO: implement
        return self.x.shape[0]
        raise NotImplementedError

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        # TODO: implement
        return self.x[idx], self.y[idx]
        raise NotImplementedError

In [ ]:
class NextTokenDataset(Dataset):
    """
    Next-token prediction dataset.

    Given tokens of shape (N, T), produce: # 下面两个是错位构成next-token prediction预测
      input_ids  = tokens[:, :-1]
      target_ids = tokens[:, 1:]

    Return per item:
      (input_ids, target_ids)

    Notes:
    - Returned tensors should be 1D of length (T-1).
    - dtype should remain integer.
    """

    def __init__(self, tokens: torch.Tensor):
        # TODO: implement
        if tokens.ndim != 2 or tokens.shape[1]<2:
            raise ValueError("tokens must have shape (N,T), with T >= 2")
        self.tokens = tokens


    def __len__(self) -> int:
        # TODO: implement
        return self.tokens.shape[0]
        raise NotImplementedError

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        # TODO: implement
        return self.tokens[idx,:-1],self.tokens[idx,1:]
        raise NotImplementedError

In [ ]:

class RandomCropSequenceDataset(Dataset):
    """
    Sequence dataset that returns random crops of fixed length.

    tokens: (N, T_total)
    crop_len: L

    For each __getitem__:
      - sample a start index s so that s+L <= T_total
      - return tokens[idx, s:s+L]

    Requirements:
    - Use a torch.Generator for deterministic behavior if seed is provided.
    - Do NOT use Python's random module.
    """

    def __init__(self, tokens: torch.Tensor, crop_len: int, seed: int | None = None):
        # TODO: implement
        if tokens.ndim !=2:
            raise ValueError("tokens must have shape (N,T_otal)")
        if not 1<= crop_len <=tokens.shape[1]:
            raise ValueError("crop_len must be between 1 and T_totals")
        self.tokens = tokens
        self.crop_len = crop_len
        self.generator = torch.Generator()
        if seed is None:
            self.generator.seed()
        else:
            self.generator.manual_seed(seed)


    def __len__(self) -> int:
        # TODO: implement
        return self.tokens.shape[0]
        raise NotImplementedError

    def __getitem__(self, idx: int) -> torch.Tensor:
        # TODO: implement
        start = torch.randint(
            self.tokens.shape[1]-self.crop_len+1, # 上界不包含，因此需要加1
            (1,), # 单元素元组，表示生成的 size
            generator=self.generator,
        ).item() # .item() 把单元素张量转成 Python 标量（int / float），方便后面直接当索引用：如果 size=(3,)，.item() 会报错，得用 .tolist() 或索引。
        return self.tokens[idx,start:start+self.crop_len]
        raise NotImplementedError

In [28]:
@dataclass(frozen=True)
class PaddedBatch:
    """
    A padded batch for variable-length sequences.

    tokens: LongTensor (B, T_max)
    lengths: LongTensor (B,)
    padding_mask: BoolTensor (B, T_max) where True means "this is padding"
    """

    tokens: torch.Tensor
    lengths: torch.Tensor
    padding_mask: torch.Tensor


def pad_1d_sequences(seqs: list[torch.Tensor], pad_value: int = 0) -> PaddedBatch:
    """
    Pad a list of 1D integer tensors to the same length.

    Requirements:
    - Return PaddedBatch(tokens, lengths, padding_mask)
    - padding_mask[b, t] == True if t >= lengths[b]
    - tokens should be dtype long, if not cast them
    """
    # TODO: implement
    if not seqs:
        return PaddedBatch(
            tokens=torch.empty((0,0),dtype=torch.long),
            lengths=torch.empty((0,),dtype=torch.long),
            padding_mask=torch.empty((0,0),dtype=torch.bool)
        )
    device = seqs[0].device
    if any(seq.ndim != 1 for seq in seqs):
        raise ValueError("each sequence must be 1D")
    if any(seq.device != device for seq in seqs):
        raise ValueError("all sequences must be on the same device")
    
    lengths = torch.tensor(
        [seq.numel() for seq in seqs],dtype=torch.long,device=device
    ) # seq.numel得到1D tensor的size
    max_len = max(seq.numel() for seq in seqs)
    tokens = torch.full(
        (len(seqs),max_len),pad_value,dtype = torch.long,device=device
    )# 预先造一个全pad矩阵
    for row,seq in enumerate(seqs):
        tokens[row,:seq.numel()] = seq.to(dtype=torch.long)
    
    position = torch.arange(max_len,device=device) # 生成序列号 0 1 2 ... max_length -1 也就是t
    padding_mask = position.unsqueeze(0) >= lengths.unsqueeze(1) # 计算 padding_mask[b, t] == True if t >= lengths[b]
    return PaddedBatch(tokens ,lengths,padding_mask)
    raise NotImplementedError

In [ ]:
def collate_next_token_batch(
    batch: list[tuple[torch.Tensor, torch.Tensor]], pad_value: int = 0
) -> dict[str, torch.Tensor]:
    """
    Collate for NextTokenDataset samples that may have variable lengths.

    batch: list of (input_ids, target_ids), each 1D

    Return dict with:
      - input_ids: (B, T_max)
      - target_ids: (B, T_max)
      - attention_mask: (B, T_max) where True means "keep" (NOT padding)
      - padding_mask: (B, T_max) where True means "padding"

    Requirements:
    - pad input_ids and target_ids consistently
    - attention_mask is the logical NOT of padding_mask
    """
    # TODO: implement
    for input_ids, target_ids in batch:
        if input_ids.ndim != 1 or target_ids.ndim!=1:
            raise ValueError("input_ids and target_ids must be 1D")
        if input_ids.numel()!=target_ids.numel():
            raise ValueError("input_ids and target_ids must have equal lengths")
        if input_ids.device != target_ids.device:
            raise ValueError("input_ids and target_ids must share a device")   
    inputs = pad_1d_sequences([pair[0] for pair in batch],pad_value)
    targets = pad_1d_sequences([pair[1] for pair in batch],pad_value)
    return {
      "input_ids": inputs.tokens,
      "target_ids": targets.tokens,
      "attention_mask": ~inputs.padding_mask, #逻辑取反用～不用-
      "padding_mask": inputs.padding_mask
    }
    raise NotImplementedError

In [30]:
def make_dataloader(
    dataset: Dataset,
    batch_size: int,
    shuffle: bool = True,
    drop_last: bool = False,
    collate_fn=None,
    num_workers: int = 0,
) -> DataLoader:
    """
    Create a DataLoader with optional collate_fn.
    """
    # TODO: implement
    return DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        collate_fn=collate_fn,
        num_workers=num_workers,
    )
    raise NotImplementedError

## Optimizers (AdamW from scratch)

PyTorch optimizers keep **state** for each parameter (e.g. moment estimates in Adam).
In this section you’ll implement **AdamW**, which is Adam + *decoupled* weight decay.

### AdamW state
For each parameter tensor `p` we store:
- `m`: first moment (EMA of gradients)
- `v`: second moment (EMA of squared gradients)
- `t`: step counter

### Update overview (high level)
1) Update moments `m, v`
2) Bias-correct them (`m_hat, v_hat`)
3) Apply parameter update:
   `p -= lr * ( m_hat / (sqrt(v_hat) + eps) + weight_decay * p )`

Notes:
- This update is **in-place** (mutates `p`).
- Gradients should not be modified.
- State tensors must match parameter shape/device/dtype.

In [31]:
@dataclass
class AdamWState:
    """
    Per-parameter AdamW state.

    m: first moment
    v: second moment
    t: step count
    """

    m: torch.Tensor
    v: torch.Tensor
    t: int


def init_adamw_state(p: torch.Tensor) -> AdamWState:
    """
    Initialize AdamW state tensors for a parameter tensor p.

    What to create:
    - m: zeros like p, same shape/device/dtype
    - v: zeros like p, same shape/device/dtype
    - t: step counter starting at 0

    Notes / requirements:
    - Use torch.zeros_like(p) for m and v.
    - Do NOT attach gradients to the state (initialize under torch.no_grad()).
    - t starts at 0. In adamw_step_, increment t to 1 on the first update *before*
      computing bias correction terms (1 - beta1^t) and (1 - beta2^t).
    - State tensors must live on the same device as p (CPU vs GPU) and have the
      same dtype as p.
    """
    # TODO: implement
    with torch.no_grad():
      return AdamWState(
        m=torch.zeros_like(p),
        v=torch.zeros_like(p),
        t=0
      )
    raise NotImplementedError

In [32]:
def adamw_step_(
    p: torch.Tensor,
    grad: torch.Tensor,
    state: AdamWState,
    lr: float,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-8,
    weight_decay: float = 0.01,
) -> AdamWState:
    """
    In-place(原地) AdamW parameter update (updates p).

    Algorithm (AdamW):
      m = beta1*m + (1-beta1)*grad
      v = beta2*v + (1-beta2)*grad^2
      m_hat = m / (1 - beta1^t)
      v_hat = v / (1 - beta2^t)
      p = p - lr * (m_hat / (sqrt(v_hat) + eps) + weight_decay * p)

    Requirements:
    - Update p in-place.
    - Return updated state (with incremented t).
    - Do not modify grad.
    - Should work for any tensor shape.
    """
    # TODO: implement
    beta1,beta2 = betas
    with torch.no_grad():
        state.t +=1
        state.m.mul_(beta1).add_(grad,alpha=1-beta1)
        state.v.mul_(beta2).addcmul_(grad,grad,value=1-beta2)

        m_hat = state.m / (1 - beta1 ** state.t)
        v_hat = state.v / (1 - beta2 ** state.t)

        p.mul_(1 - lr * weight_decay)
        p.addcdiv_(m_hat, v_hat.sqrt().add_(eps), value=-lr)
    return state
    raise NotImplementedError

In [33]:
def adamw_step_many_(
    params: list[torch.Tensor],
    grads: list[torch.Tensor],
    states: list[AdamWState],
    lr: float,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-8,
    weight_decay: float = 0.01,
) -> list[AdamWState]:
    """
    Apply AdamW to many parameters.

    Requirements:
    - len(params) == len(grads) == len(states)
    - Update each param in-place.
    - Return the list of updated states.
    """
    # TODO: implement

    if not (len(params) == len(grads) == len(states)):
        raise ValueError("params, grads, and states must have equal lengths")
    for p, grad, state in zip(params, grads, states):
        adamw_step_(p, grad, state, lr, betas, eps, weight_decay)
    return states

    raise NotImplementedError

## Training basics

A minimal training step follows the same pattern almost everywhere:

1) set model to train mode
2) reset gradients
3) forward pass
4) compute loss
5) backward pass
6) step optimizer

In this exercise you’ll implement a single MSE training step using a standard PyTorch optimizer.
Return a Python float loss value.

In [ ]:
def train_step_mse(
    model: nn.Module,
    batch: tuple[torch.Tensor, torch.Tensor],
    optimizer: torch.optim.Optimizer,
) -> float:
    """
    One MSE train step using standard torch optimizer.
    """
    # TODO: implement
    model.train()
    optimizer.zero_grad(set_to_none=True) # 不清零、直接扔掉梯度张量
    x,y=batch
    prediction = model(x)
    loss = nn.functional.mse_loss(prediction,y)
    loss.backward()
    optimizer.step()
    return loss.detach().item()
    raise NotImplementedError


## Parameter initialization

Initialization matters because it controls signal and gradient scales at the start of training.

### Fan-in / fan-out
- `fan_in`: number of input connections to a unit
- `fan_out`: number of output connections from a unit

For a Linear layer weight of shape `(out_features, in_features)`:
- `fan_in = in_features`
- `fan_out = out_features`

### Common schemes
- **Xavier / Glorot** (often good for tanh / linear-ish nets):
  keeps variance stable across layers when activations are roughly symmetric.
- **Kaiming / He** (often good for ReLU-like nets):
  accounts for the fact that ReLU zeroes out about half the inputs.

In this section you’ll implement Xavier uniform and Kaiming uniform and use them to initialize `nn.Linear`.
We also always zero the bias unless explicitly told otherwise.

In [35]:
def fan_in_fan_out(weight: torch.Tensor) -> tuple[int, int]:
    """Compute (fan_in, fan_out) for a weight tensor."""
    # TODO: implement

    if weight.ndim < 2:
        raise ValueError("weight must have at least 2 dimensions")
    receptive_field_size = 1
    for size in weight.shape[2:]:
        receptive_field_size *= size
    fan_in = weight.shape[1] * receptive_field_size
    fan_out = weight.shape[0] * receptive_field_size
    return fan_in, fan_out
    raise NotImplementedError

In [36]:
def xavier_uniform_(weight: torch.Tensor, gain: float = 1.0) -> torch.Tensor:
    """
    In-place Xavier/Glorot uniform init:
      bound = gain * sqrt(6 / (fan_in + fan_out))
      U(-bound, bound)
    """
    # TODO: implement

    fan_in, fan_out = fan_in_fan_out(weight)
    if weight.numel() == 0:
        return weight
    bound = gain * (6.0 / (fan_in + fan_out)) ** 0.5
    with torch.no_grad():
        weight.uniform_(-bound, bound)
    return weight
    raise NotImplementedError

In [37]:
def kaiming_uniform_(weight: torch.Tensor, nonlinearity: str = "relu") -> torch.Tensor:
    """
    In-place Kaiming/He uniform init.

    Follow this common choice:
      gain = sqrt(2) for ReLU
      std = gain / sqrt(fan_in)
      bound = sqrt(3) * std
      U(-bound, bound)
    """
    # TODO: implement

    fan_in, _ = fan_in_fan_out(weight)
    if weight.numel() == 0:
        return weight
    gain = nn.init.calculate_gain(nonlinearity)
    std = gain / fan_in ** 0.5
    bound = 3.0 ** 0.5 * std
    with torch.no_grad():
        weight.uniform_(-bound, bound)
    return weight

    raise NotImplementedError

In [38]:
def init_linear_(layer: nn.Linear, scheme: str = "xavier") -> nn.Linear:
    """
    Initialize an nn.Linear in-place.

    scheme:
      - "xavier"
      - "kaiming_relu"
      - "zero" (weights and bias = 0)
    """
    # TODO: implement
    with torch.no_grad():
        if scheme == "xavier":
            xavier_uniform_(layer.weight)
        elif scheme == "kaiming_relu":
            kaiming_uniform_(layer.weight,nonlinearity="relu")
        elif scheme == "zero":
            layer.weight.zero_()
        else:
            raise ValueError(f"Unknown initialization scheme: {scheme}")
        if layer.bias is not None:
            layer.bias.zero_()
            
    return layer

    raise NotImplementedError